# 第8回講義 宿題



## 課題
ViTによる画像分類を実装してみましょう．


### 目標値
なし
- 今回は計算リソースによってモデルの性能が大きく変わるため，目標精度は設定していません．

### ルール
- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- **演習用に配布されている`trained_vision_model.pth`は使わないでください．**


### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルを`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第8回 Transformer基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第8回 Transformer基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．


### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 作業ディレクトリを指定
work_dir = '/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab'

### データの読み込み（このセルは修正しないでください）

In [ ]:
!sudo apt update
!sudo apt install xvfb

import sys
import random
import math
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
import h5py
from os.path import join
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from torchvision import datasets, transforms
from einops.layers.torch import Rearrange
from einops import rearrange, repeat

import logging

import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.nn import functional as F

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.set_printoptions(edgeitems=1e3)

# 要素にドットでアクセスできる辞書クラス
class Args(dict):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.__dict__ = self

#学習データ
x_train = np.load(work_dir + '/Lecture08/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture08/data/t_train.npy')

#テストデータ
x_test = np.load(work_dir + '/Lecture08/data/x_test.npy')

class train_dataset(torch.utils.data.Dataset):
    def __init__(self, x_train, t_train):
        data = x_train.astype('float32')
        self.x_train = []
        for i in range(data.shape[0]):
            self.x_train.append(Image.fromarray(np.uint8(data[i])))
        self.t_train = t_train
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_train)

    def __getitem__(self, idx):
        return self.transform(self.x_train[idx]), torch.tensor(t_train[idx], dtype=torch.long)

class test_dataset(torch.utils.data.Dataset):
    def __init__(self, x_test):
        data = x_test.astype('float32')
        self.x_test = []
        for i in range(data.shape[0]):
            self.x_test.append(Image.fromarray(np.uint8(data[i])))
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_test)

    def __getitem__(self, idx):
        return self.transform(self.x_test[idx])

trainval_data = train_dataset(x_train, t_train)
test_data = test_dataset(x_test)

### データセットの準備  

In [ ]:
val_size = 3000
train_data, valid_data = torch.utils.data.random_split(trainval_data, [len(trainval_data) - val_size, val_size])

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))]
)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))]
)

trainval_data.transform = train_transform
test_data.transform = test_transform

### ViTの実装

In [ ]:
# ViTの実装

class SelfAttention(nn.Module):
  # WRITE ME


class Block(nn.Module):
  # WRITE ME


class PatchEmbedding(nn.Module):
  # WRITE ME


class ViT(nn.Module):
  # WRITE ME


In [ ]:
# Trainerを設定

logger = logging.getLogger(__name__)

class TrainerConfig:
    # 最適化のパラメータ
    max_epochs = 10
    batch_size = 64
    learning_rate = 3e-4
    betas = (0.9, 0.95)
    grad_norm_clip = 1.0
    weight_decay = 0.1  # 行列乗算に使用する重みにのみ適用
    # 学習率の減衰パラメータ：線形warmupの後、元の学習率の10%までcosine減衰
    lr_decay = False
    warmup_tokens = 375e6  # warmup_tokensとfinal_tokensの値はGPT-3論文に由来するが，他のケースでも適切な初期値とは限らない
    final_tokens = 260e9  # このトークン数を処理した時点で，学習率が始めの値の10%まで下がるようにする
    # チェックポイントの設定
    ckpt_path = None
    num_workers = 0  # DataLoader用

    def __init__(self, **kwargs):
        for k,v in kwargs.items():
            setattr(self, k, v)

class Trainer:

    def __init__(self, model, train_dataset, test_dataset, config):
        self.model = model
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.config = config

        # システム上にあるすべてのGPUを使用
        self.device = 'cpu'
        if torch.cuda.is_available():
            self.device = torch.cuda.current_device()
            self.model = torch.nn.DataParallel(self.model).to(self.device)

    def save_checkpoint(self):
        # DataParallel Wrapperは生のモデルのオブジェクトを.moduleに保持する
        raw_model = self.model.module if hasattr(self.model, "module") else self.model
        logger.info("saving %s", self.config.ckpt_path)
        torch.save(raw_model.state_dict(), self.config.ckpt_path)

    def train(self):
        model, config = self.model, self.config
        raw_model = model.module if hasattr(self.model, "module") else model
        optimizer = raw_model.configure_optimizers(config)

        def run_epoch(split):
            is_train = split == 'train'
            model.train(is_train)
            data = self.train_dataset if is_train else self.test_dataset
            shuffle = is_train
            loader = DataLoader(data, shuffle=shuffle, pin_memory=True,
                                batch_size=config.batch_size,
                                num_workers=config.num_workers)

            losses = []
            pbar = tqdm(enumerate(loader), total=len(loader)) if is_train else enumerate(loader)
            for it, (x, y) in pbar:

                # データを適切なデバイスに配置
                x = x.to(self.device)
                y = y.to(self.device)

                # 順伝播
                with torch.set_grad_enabled(is_train):
                    logits, loss = model(x, y)
                    loss = loss.mean()  # 複数GPUに分散している場合損失をまとめる
                    losses.append(loss.item())

                if is_train:

                    # 逆伝播およびパラメータ更新
                    model.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_norm_clip)
                    optimizer.step()

                    # 進捗に応じて学習率を減衰
                    if config.lr_decay:
                        self.tokens += (y >= 0).sum()  # このステップで処理されたトークン数（ラベルが -100 でないもの）
                        if self.tokens < config.warmup_tokens:
                            # 線形warmup
                            lr_mult = float(self.tokens) / float(max(1, config.warmup_tokens))
                        else:
                            # cosine学習率減衰
                            progress = float(self.tokens - config.warmup_tokens) / float(max(1, config.final_tokens - config.warmup_tokens))
                            lr_mult = max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))
                        lr = config.learning_rate * lr_mult
                        for param_group in optimizer.param_groups:
                            param_group['lr'] = lr
                    else:
                        lr = config.learning_rate

                    # 進捗の表示
                    pbar.set_description(f"epoch {epoch+1} iter {it}: train loss {loss.item():.5f}. lr {lr:e}")

            if not is_train:
                test_loss = float(np.mean(losses))
                logger.info("test loss: %f", test_loss)
                return test_loss

        best_loss = float('inf')
        self.tokens = 0  # 学習率減衰に使用するカウンタ
        for epoch in range(config.max_epochs):

            run_epoch('train')
            if self.test_dataset is not None:
                test_loss = run_epoch('test')

            # テストlossに基づく早期終了・テストを行わない場合は常にチェックポイントを保存
            good_model = self.test_dataset is None or test_loss < best_loss
            if self.config.ckpt_path is not None and good_model:
                best_loss = test_loss
                self.save_checkpoint()

In [ ]:
block_size = 256

args = Args({
    # WRITE ME
})

model = ViT(args)  # あとでtrainerがモデルをGPUに移してくれる

# Trainerをインスタンス化し, 訓練を開始
tconf = TrainerConfig(
    # WRITE ME
)

trainer = Trainer(model, train_data, valid_data, tconf)

model_path = work_dir + '/Lecture08/models/trained_vision_model_homework.pth'

In [ ]:
# 学習
trainer.train()

# 学習したパラメータの保存
torch.save(model.state_dict(), model_path)

In [ ]:
# 評価の準備
device = "cuda" if torch.cuda.is_available() else "cpu"

# 学習したパラメータの読み込み
model.load_state_dict(torch.load(model_path))
model.eval();

In [ ]:
# datasetをdata loaderにする
train_dataloader = DataLoader(train_data, shuffle=True, pin_memory=True,
                              batch_size=tconf.batch_size, num_workers=tconf.num_workers)
valid_dataloader = DataLoader(valid_data, shuffle=False, pin_memory=True,
                             batch_size=tconf.batch_size, num_workers=tconf.num_workers)

train_acc, valid_acc = 0., 0.
with torch.no_grad():
    for x, y in train_dataloader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x, y)

        acc = (torch.argmax(logits, dim=1) == y).float().sum().cpu()
        train_acc += acc

    for x, y in valid_dataloader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x, y)

        acc = (torch.argmax(logits, dim=1) == y).float().sum().cpu()
        valid_acc += acc

print(f"Train Acc.: {(train_acc / len(train_data)):.4f}")
print(f"Valid Acc. : {(valid_acc / len(valid_data)):.4f}")

In [ ]:
test_dataloader = DataLoader(test_data, shuffle=False, pin_memory=True,
                             batch_size=tconf.batch_size, num_workers=tconf.num_workers)

t_pred = []
with torch.no_grad():
    for x in test_dataloader:
        x = x.to(device)
        logits, _ = model(x, None)

        # モデルの出力を予測値のスカラーに変換
        pred = logits.argmax(1).tolist()
        t_pred.extend(pred)

submission = pd.Series(t_pred, name='label')
submission.to_csv(work_dir + '/Lecture08/submission_pred.csv', header=True, index_label='id')